# DLTS — Deep Learning assisted Tree SearchPort de **DLTS-DFS** y **DLTS-LDS** (Hottung, Tanaka & Tierney, 2020 — *Deep learningassisted heuristic tree search for the container pre-marshalling problem*) alimentadopor los modelos de este framework.La idea: en vez de decidir el movimiento de forma **greedy** (tomar siempre el logitmás alto) o de mantener un **haz** de estados (beam search), DLTS hace una búsqueda enárbol donde la red de política decide **qué ramas vale la pena abrir**:- En cada nodo se consulta la política, se descartan los movimientos ilegales y se  ramifica sólo sobre los movimientos cuya probabilidad supera un umbral que depende  de la probabilidad máxima del nodo y de la profundidad (`bstrategy`, `p`).- **DFS** explora en profundidad con backtracking; **LDS** (*limited discrepancy  search*) explora por número de "discrepancias" respecto de la decisión greedy,  usando una cola de prioridad.- Opcionalmente, el *cost model* actúa como **cota inferior** para podar  (`use_bounding`); ver la sección de parámetros, donde se explica por qué aquí está  apagado por defecto.Ambas estrategias son **anytime**: mantienen la mejor solución encontrada(*incumbent*) y siguen mejorándola hasta agotar el presupuesto.

## 1. SetupEl paquete vive en `src/`. Si el proyecto está instalado (`pip install -e .`) losimports funcionan tal cual; si no, esta celda añade `src/` al path.`torch.set_num_threads(1)` es importante para que los tiempos sean comparables entresolvers: sin eso, el greedy (que hace forwards de batch grande) aprovecha más hilosque DLTS (un forward por nodo) y la comparación de tiempos deja de significar nada.

In [ ]:
import sysfrom pathlib import Pathtry:    import solvers  # noqa: F401except ModuleNotFoundError:    SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"    sys.path.insert(0, str(SRC))import torchtorch.set_num_threads(1)print("torch", torch.__version__)

## 2. Cargar los modelosDos redes distintas, con papeles distintos:- **Red de branching** (`action_model`): la política. Da la distribución sobre  movimientos que decide qué ramas se abren. Se puede usar `sl` (entrenada por  imitación) o `rl` (afinada con REINFORCE) — cambiar `ACTION_MODEL` y volver a  ejecutar es todo lo que hace falta para comparar.- **Red de bounding** (`cost_model`): el *cost predictor*. Estima el coste restante  y sólo se usa si `use_bounding=True`.El adaptador de entrada **debe ser el mismo que se usó en el entrenamiento**:`EnrichedLayoutAdapter(Layout4DAdapterV1, StackFeaturesAdapterV1, S, H)`, igual que en`scripts/benchmarking_bs_cost.py`. Ojo con `S_max`/`H_max`: en este framework seconstruyen **por tamaño de instancia** (`S_max = S`, `H_max = H`), no con un máximoglobal, así que hay que rehacer el adaptador al cambiar de carpeta de benchmark.

In [ ]:
from models.actions.v1 import CPMPTransformerfrom models.cost.v1 import CostPredictorTransformerfrom training.common import load_modelfrom data.adapters.input import EnrichedLayoutAdapter, Layout4DAdapterV1, StackFeaturesAdapterV1ACTION_MODEL = "rl"     # "sl" | "rl"  -> red de branching (la política)COST_MODEL   = "cost"   # red de bounding (sólo se usa con use_bounding=True)action_model = load_model(CPMPTransformer, ACTION_MODEL)cost_model   = load_model(CostPredictorTransformer, COST_MODEL)# Instancia de trabajo para las secciones 3 y 4S, H, MAX_STEPS = 5, 7, 100FOLDER   = "benchmarks/5-5"INSTANCE = "benchmarks/5-5/data5-5-1.dat"input_adapter = EnrichedLayoutAdapter(Layout4DAdapterV1, StackFeaturesAdapterV1, S, H)print(f"branching = {ACTION_MODEL} | bounding = {COST_MODEL} | S={S} H={H}")

## 3. Una instancia: DFS y LDS`DLTSSolver` implementa la interfaz `Solver` del framework, así que `solve`,`solve_from_layout` y `solve_from_folder` funcionan igual que con `ModelSolver` o`BSGCostPredictorSolver`.### El presupuesto (`NodeBudget`)El paper original corta la búsqueda **por tiempo**. Aquí el criterio principal es un**presupuesto de nodos** (`max_expansions`), porque el tiempo de un forward dePyTorch no es comparable con el de la red Keras/Theano del paper y porque un tope denodos hace las comparaciones reproducibles entre máquinas. `NodeBudget` aceptatambién `timeout_s` si se prefiere el criterio del paper, más `max_nn_calls`,`max_depth` y `max_open_nodes` como topes secundarios.`stop_reason` dice cómo terminó: `exhausted` = agotó el árbol (ya no queda nada porexplorar, subir el presupuesto no cambiaría nada), `budget` = se quedó sinpresupuesto, `timeout`, `open_list_overflow`, `already_sorted` = la instancia veníaresuelta.

In [ ]:
from solvers import DLTSSolver, ModelSolverfrom solvers.dlts.counters import NodeBudgetbudget = NodeBudget(max_expansions=2048)# Referencia: la misma política, sin búsquedasolved, steps, t = ModelSolver(action_model, input_adapter).solve(INSTANCE, H, MAX_STEPS)print(f"{'greedy':10} resuelto={solved} pasos={steps} tiempo={t:.3f}s")for strategy in ("dfs", "lds"):    solver = DLTSSolver(        action_model, input_adapter, cost_model=cost_model,        strategy=strategy, p=0.20, bstrategy="log", budget=budget,    )    solved, steps, t = solver.solve(INSTANCE, H, MAX_STEPS)    st = solver.last_stats[-1]    print(f"{solver.name:10} resuelto={solved} pasos={steps} tiempo={t:.3f}s "          f"| nodos={st['nodes_expanded']} llamadas_red={st['nn_calls']} fin={st['stop_reason']}")

### Métricas por instancia`solver.last_stats` acumula un diccionario por instancia resuelta. Las claves que másse usan:| clave | qué es ||---|---|| `nodes_expanded` | nodos procesados (el "Avg. Opened Nodes" del paper) || `nodes_generated` | hijos creados || `nn_calls` | consultas a las redes (política + valor) || `open_list_max` | pico de la frontera (pila de DFS / heap de LDS) || `depth_max` | profundidad máxima alcanzada || `incumbent_updates` | veces que se mejoró la mejor solución || `stop_reason` | `exhausted` / `budget` / `timeout` / `already_sorted` || `anytime` | curva anytime: `[(nodos, mejor_coste), ...]` |**`nodes_expanded` no es comparable con el de beam search**: un nodo de DLTS es *una*consulta a la política, mientras que un nodo de beam expande un estado y evalúa**todos** sus hijos con el cost model (decenas de forwards). Para comparar métodosdistintos hay que mirar el **tiempo**, o `nn_calls`, no los nodos.La curva `anytime` permite leer qué solución se tenía a cualquier presupuesto menor**sin volver a correr la búsqueda**: el presupuesto sólo corta el bucle, no cambianinguna decisión, así que una corrida al presupuesto máximo contiene a todas las máscortas. Hay que pedirla explícitamente pasando los cortes en `NodeBudget(checkpoints=...)`.

In [ ]:
CORTES = (32, 64, 128, 256, 512, 1024, 2048, 4096)solver = DLTSSolver(action_model, input_adapter, cost_model=cost_model,                    strategy="lds", p=0.20, bstrategy="log",                    budget=NodeBudget(max_expansions=4096, checkpoints=CORTES))solver.solve(INSTANCE, H, MAX_STEPS)st = solver.last_stats[-1]for k in ("nodes_expanded", "nodes_generated", "nn_calls", "open_list_max",          "depth_max", "incumbent_updates", "stop_reason", "time_s"):    print(f"{k:18} {st[k]}")print("\ncurva anytime (nodos -> mejor coste):")for nodes, cost in st["anytime"]:    print(f"  {nodes:>6} nodos -> {cost} pasos")print("  (si la búsqueda se agota entre dos cortes, la última mejora no queda en la"      " curva: el valor final es el de la primera línea de arriba)")

## 4. Comparación en una carpetaLos cuatro métodos sobre las 40 instancias de `benchmarks/5-5`, con el **mismo**modelo debajo — así lo que se compara es la *estrategia de búsqueda*, no la red.Tarda un par de minutos. Bajar `max_expansions` para que vaya más rápido.

In [ ]:
import statistics, timefrom solvers import BSGCostPredictorSolverdef evaluar(nombre, solver):    t0 = time.perf_counter()    resultados = solver.solve_from_folder(FOLDER, H, MAX_STEPS)    dt = time.perf_counter() - t0    ok = [r for r in resultados if r[0]]    pasos = statistics.mean(r[1] for r in ok) if ok else float("nan")    return {"metodo": nombre, "resueltas": f"{len(ok)}/{len(resultados)}",            "pasos_medios": round(pasos, 2), "tiempo_total_s": round(dt, 1)}filas = [    evaluar("greedy", ModelSolver(action_model, input_adapter)),    evaluar("beam w=32", BSGCostPredictorSolver(action_model, cost_model, input_adapter, 32)),]for strategy in ("dfs", "lds"):    filas.append(evaluar(        f"DLTS-{strategy.upper()}",        DLTSSolver(action_model, input_adapter, cost_model=cost_model,                   strategy=strategy, p=0.20, bstrategy="log",                   budget=NodeBudget(max_expansions=2048)),    ))ancho = {k: max(len(k), max(len(str(f[k])) for f in filas)) for k in filas[0]}print(" | ".join(k.ljust(ancho[k]) for k in filas[0]))print("-+-".join("-" * ancho[k] for k in filas[0]))for f in filas:    print(" | ".join(str(f[k]).ljust(ancho[k]) for k in f))

## 5. Parámetros### Ramificación- **`bstrategy`** — cómo se afloja el umbral con la profundidad: `constant`,  `linear`, `quadratic`, `log`. Son las cuatro funciones MP del paper (§4.4).- **`p`** — cuán agresiva es la poda. Es el parámetro que más manda: más alto = se  abren más ramas = mejores soluciones y más nodos.- **`temperature`** — reescala los logits antes del softmax. `1.0` deja la política  tal como fue entrenada.### Bounding (apagado por defecto)`use_bounding=True` usa el cost model para podar: si `d · V̂(estado) + coste_actual`supera el incumbent, la rama se corta. `d` compensa el sesgo del estimador y `n` esla profundidad a partir de la cual se aplica.**Está apagado a propósito.** El cost model se entrena sobre las trayectorias de lapropia política, así que estima `V^π` (lo que *nuestra política* tardaría) y no `V*`(el óptimo). Como `V^π ≥ V*`, `d·V̂` **no es una cota inferior válida** y puede podarla rama que contenía la mejor solución. En nuestras mediciones el bounding empeorabael resultado de forma consistente. Se deja conmutable como ablación.### Advertencia sobre los valores calibradosLos valores de abajo (`p=0.20`, `bstrategy="log"`, `temperature=1.0`, y `d=0.624`,`n=5` para el bounding) se calibraron contra **otro modelo**, no contra loscheckpoints de este repo. Sirven como punto de partida razonable, no como el óptimopara `sl`/`rl` de aquí: conviene rebarrer al menos `p`, que es el que domina.La celda siguiente hace ese barrido sobre una carpeta.

In [ ]:
PARAMS = dict(bstrategy="log", p=0.20, temperature=1.0, use_bounding=False)PARAMS_BOUNDING = dict(bstrategy="log", p=0.20, temperature=1.0,                       use_bounding=True, d=0.624, n=5)# Barrido de p: calidad contra esfuerzo. Subir p siempre mejora los pasos;# la pregunta es a qué precio en nodos.for p in (0.05, 0.10, 0.20, 0.30):    solver = DLTSSolver(action_model, input_adapter, cost_model=cost_model,                        strategy="lds", bstrategy="log", p=p,                        budget=NodeBudget(max_expansions=2048))    resultados = solver.solve_from_folder(FOLDER, H, MAX_STEPS)    ok = [r for r in resultados if r[0]]    nodos = statistics.mean(s["nodes_expanded"] for s in solver.last_stats)    agotados = sum(s["stop_reason"] == "exhausted" for s in solver.last_stats)    print(f"p={p:<5} resueltas {len(ok)}/{len(resultados)} | "          f"pasos {statistics.mean(r[1] for r in ok):.2f} | "          f"nodos {nodos:.0f} | arbol agotado en {agotados}/{len(resultados)}")

Si `arbol agotado` sale alto, la búsqueda **termina antes de gastar el presupuesto**:ahí subir `max_expansions` no cambia nada y el único modo de mejorar es subir `p`.## 6. Barrido de los 19 benchmarks (lento — ejecutar a voluntad)Recorre todas las carpetas con DFS y LDS y escribe un CSV por configuración en`experiments/`. Cada carpeta reconstruye su propio adaptador con `S_max = S` y`H_max = H`.En un portátil son **horas**. Para una pasada rápida, recortar `BENCHMARK_CONFIG` obajar `max_expansions`.

In [ ]:
# H = pisos + 2, con el max_steps por grupo de tamaño.BENCHMARK_CONFIG = {    "benchmarks/3-3":  {"S": 3,  "H": 5, "max_steps": 50},    "benchmarks/3-4":  {"S": 4,  "H": 5, "max_steps": 50},    "benchmarks/3-5":  {"S": 5,  "H": 5, "max_steps": 50},    "benchmarks/3-6":  {"S": 6,  "H": 5, "max_steps": 50},    "benchmarks/3-7":  {"S": 7,  "H": 5, "max_steps": 50},    "benchmarks/3-8":  {"S": 8,  "H": 5, "max_steps": 50},    "benchmarks/4-4":  {"S": 4,  "H": 6, "max_steps": 50},    "benchmarks/4-5":  {"S": 5,  "H": 6, "max_steps": 50},    "benchmarks/4-6":  {"S": 6,  "H": 6, "max_steps": 50},    "benchmarks/4-7":  {"S": 7,  "H": 6, "max_steps": 50},    "benchmarks/5-4":  {"S": 4,  "H": 7, "max_steps": 100},    "benchmarks/5-5":  {"S": 5,  "H": 7, "max_steps": 100},    "benchmarks/5-6":  {"S": 6,  "H": 7, "max_steps": 100},    "benchmarks/5-7":  {"S": 7,  "H": 7, "max_steps": 100},    "benchmarks/5-8":  {"S": 8,  "H": 7, "max_steps": 100},    "benchmarks/5-9":  {"S": 9,  "H": 7, "max_steps": 100},    "benchmarks/5-10": {"S": 10, "H": 7, "max_steps": 100},    "benchmarks/6-6":  {"S": 6,  "H": 8, "max_steps": 150},    "benchmarks/6-10": {"S": 10, "H": 8, "max_steps": 150},}

In [ ]:
import csv, osfrom settings import INSTANCE_FOLDER, EXPERIMENTS_FOLDERfrom cpmp.layout import read_filedef barrer(strategy, max_expansions=2048, config=BENCHMARK_CONFIG, **kwargs):    """Corre DLTS sobre las carpetas de `config` y escribe un CSV por corrida."""    EXPERIMENTS_FOLDER.mkdir(parents=True, exist_ok=True)    salida = EXPERIMENTS_FOLDER / f"DLTS-{strategy.upper()}_{ACTION_MODEL}_b{max_expansions}.csv"    with open(salida, "w", newline="") as fh:        writer = csv.writer(fh)        writer.writerow(["Folder", "Instance", "Solved", "Steps", "Time",                         "Nodes", "NNCalls", "StopReason"])        for folder, cfg in config.items():            S_f, H_f, max_steps = cfg["S"], cfg["H"], cfg["max_steps"]            adapter = EnrichedLayoutAdapter(Layout4DAdapterV1, StackFeaturesAdapterV1, S_f, H_f)            solver = DLTSSolver(action_model, adapter, cost_model=cost_model,                                strategy=strategy,                                budget=NodeBudget(max_expansions=max_expansions),                                **{**PARAMS, **kwargs})            ruta = INSTANCE_FOLDER / folder            nombres = sorted(os.listdir(ruta))            pasos_ok = []            for nombre in nombres:                layout = read_file(os.path.join(ruta, nombre), H_f)                solved, steps, t = solver.solve_from_layout(layout, H_f, max_steps)                st = solver.last_stats[-1]                writer.writerow([folder, nombre, solved, steps, f"{t:.4f}",                                 st["nodes_expanded"], st["nn_calls"], st["stop_reason"]])                if solved:                    pasos_ok.append(steps)            fh.flush()            media = statistics.mean(pasos_ok) if pasos_ok else float("nan")            print(f"{folder:18} resueltas {len(pasos_ok)}/{len(nombres)} | pasos {media:.2f}", flush=True)    print(f"\nGuardado en {salida}")    return salida# Descomentar para lanzarlo:# barrer("lds")# barrer("dfs")